# Batch MCQ synthesis dataset

Calls the same prompt as `scripts/pyq-llm` to produce `mcq_train.jsonl` with fields:
`messages` (user context + assistant JSON) for supervised fine-tuning.

In [ ]:
import json, httpx, os
from pathlib import Path

API = os.environ.get('PYQ_API', 'http://127.0.0.1:8787')
CHAPTERS = ['Solutions', 'Electrochemistry', 'Amines']  # extend
OUT = Path('mcq_train.jsonl')

with OUT.open('w', encoding='utf-8') as f:
    for ch in CHAPTERS:
        r = httpx.post(f'{API}/api/mcq/generate', json={'chapter': ch, 'count': 20}, timeout=120)
        r.raise_for_status()
        for item in r.json():
            row = {
                'instruction': f'Generate CBSE Class 12 Chemistry MCQ for {ch}',
                'output': json.dumps(item, ensure_ascii=False)
            }
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
print('Wrote', OUT)